In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

port = pd.read_csv(OUT / "portfolio_A_core_358.csv")
rets = pd.read_csv(DATA_PROCESSED / "return_panel_500.csv", parse_dates=["date"])
rets["ticker"] = rets["ticker"].astype(str)
port["ticker"] = port["ticker"].astype(str)

port["report_date"] = pd.to_datetime(port["report_date"])
rets["date"] = pd.to_datetime(rets["date"])

#forward returns w/ next month's realized return
pf = port.merge(rets, left_on=["report_date", "ticker"], right_on=["date", "ticker"], how="left", suffixes=("", "_r"))
pf["weighted_return"] = pf["weight"] * pf["return"]

bt = pf.groupby("report_date", as_index=False).agg(
    portfolio_return=("weighted_return", "sum"),
    n_names=("ticker", "nunique"),
    n_holdings=("selected", "sum")
)

bt["cum_return"] = (1 + bt["portfolio_return"].fillna(0)).cumprod() - 1
bt.to_csv(OUT / "backtest_A_core_358.csv", index=False)

print(bt.head())

  report_date  portfolio_return  n_names  n_holdings  cum_return
0  2020-07-31          0.047176       36          20    0.047176
1  2020-08-31          0.040727       14          14    0.089824
2  2020-09-30         -0.030642      260          20    0.056430
3  2020-10-31         -0.019540       36          20    0.035788
4  2020-11-30          0.103435       15          15    0.142925
